In [4]:
import pandas as pd

df = pd.read_csv('../data/raw/16OCLicitacion.csv', sep=';', encoding='latin-1')
df.shape

(28034, 46)

## Tratamiento de nulos en RegionUnidadCompra

La columna tenía 1.269 valores nulos. Investigando en el notebook de exploración,
se detectó que estos nulos correspondían a un grupo específico de instituciones,
no a datos faltantes aleatorios.

Se aplicó un mapeo manual basado en el nombre de cada institución:
- Instituciones con ubicación física única (hospitales, corporaciones municipales)
  fueron mapeadas a su región correspondiente.
- Instituciones de alcance nacional (ej. Servicio Nacional de Migraciones) se
  categorizaron como 'Nacional / Multiples regiones', ya que asignarles una
  única región sería inexacto.
- Un caso sin información suficiente se dejó como 'Sin especificar'.

In [8]:
mapeo_regiones = {
    'HOSPITAL COMPLEJO ASISTENCIAL PADRE LAS CASAS': 'Region de la Araucania',
    'SERVICIO LOCAL DE EDUCACIÓN PÚBLICA DE VALDIVIA': 'Region de Los Rios',
    'CORP MUNICIPAL PARA EL DESARROLLO SOCIAL': 'Region Metropolitana de Santiago',
    'Corporación de Desarrollo de La Reina': 'Region Metropolitana de Santiago',
    'Corporación Municipal de Desarrollo Productivo y Turismo de Molina': 'Region del Maule',
    'CORP MUNICIPAL DE RENCA': 'Region Metropolitana de Santiago',
    'Servicio Local de Educación Pública Iquique': 'Region de Tarapaca',
    'SERVICIO NACIONAL DE MIGRACIONES': 'Nacional / Multiples regiones',
    'SERVICIO NACIONAL DE PROTECCIÓN ESPECIALIZADA A LA NIÑEZ Y ADOLESCENCI': 'Nacional / Multiples regiones',
    'FUNDACION EDUCACIONAL PARA EL DESAROLLO INTEGRAL DE LA NIÑEZ': 'Sin especificar',
}

In [9]:
df['RegionUnidadCompra'] = df['RegionUnidadCompra'].fillna(df['Institucion'].map(mapeo_regiones))

In [10]:
df['RegionUnidadCompra'].isnull().sum()

np.int64(0)

In [11]:
df['FechaEnvioOC'].head()

0    2026-03-16
1    2026-02-05
2    2026-03-23
3    2026-03-23
4    2026-03-23
Name: FechaEnvioOC, dtype: str

## Conversión de FechaEnvioOC a tipo fecha

La columna venía como texto (str) en formato ISO (YYYY-MM-DD). Se convierte a
tipo datetime para poder extraer año, mes y trimestre en el análisis de
estacionalidad (Fase 3).

In [12]:
df['FechaEnvioOC'] = pd.to_datetime(df['FechaEnvioOC'])
df['FechaEnvioOC'].dtype

dtype('<M8[us]')

In [13]:
df['FechaEnvioOC'].dt.month.head()

0    3
1    2
2    3
3    3
4    3
Name: FechaEnvioOC, dtype: int32

## Unificación de MontoNetoOC en pesos chilenos

Las columnas MontoNetoOC_CLP e ImpuestosOC_CLP tenían ~21% de nulos. Se investigó
en el notebook de exploración y se confirmó que estos nulos correspondían a
órdenes que ya estaban en CLP (no había necesidad de conversión), por lo que el
valor ya estaba disponible en la columna sin sufijo _CLP.

Se crea una columna unificada que toma el valor convertido cuando existe, y si
no, usa el valor original (que ya está en CLP).

In [15]:
df['MontoNetoOC_final'] = df['MontoNetoOC_CLP'].fillna(df['MontoNetoOC'])
df['MontoNetoOC_final'].isnull().sum()

np.int64(0)

## Eliminación de columna Financiamiento

La columna tiene 59% de nulos, y de los valores presentes se detectaron 1.757
categorías distintas sin estandarizar (códigos internos, variantes de texto
como "SEGÚN CDP" vs "SEGUN CDP"). No aporta a ninguna de las preguntas de
negocio definidas en el proyecto, por lo que se descarta del análisis.

In [16]:
df['Financiamiento'].value_counts()

Financiamiento
PERCAPITA       998
CORPORACIÓN     706
SEGÚN CDP       303
P02             288
SEGUN CDP       190
               ... 
2161              1
31.01.002         1
2211999           1
2251              1
188               1
Name: count, Length: 1757, dtype: int64

In [17]:
df = df.drop(columns=['Financiamiento'])
df.shape

(28034, 46)

In [18]:
'Financiamiento' in df.columns

False

In [20]:
df[df['ImpuestosOC_CLP'].isnull()]['MonedaOC'].value_counts()

MonedaOC
CLP    5784
USD      16
CLF      15
Name: count, dtype: int64

## Unificación de ImpuestosOC y MontoNetoItem en pesos chilenos

Se confirmó el mismo patrón detectado en MontoNetoOC_CLP: los nulos en
ImpuestosOC_CLP corresponden mayoritariamente a órdenes ya en CLP (5.784 de
5.815), por lo que se aplica el mismo criterio de unificación.

In [22]:
df['ImpuestosOC_final'] = df['ImpuestosOC_CLP'].fillna(df['ImpuestosOC'])
df['MontoNetoItem_final'] = df['MontoNetoItemCLP'].fillna(df['MontoNetoItem'])

print(df['ImpuestosOC_final'].isnull().sum())
print(df['MontoNetoItem_final'].isnull().sum())

0
0


## Guardado del dataset procesado

Se guarda el resultado de la limpieza en data/processed/, para que quede
disponible en la Fase 3 (EDA) sin necesidad de repetir todo el proceso de
limpieza cada vez.

In [23]:
df.to_csv('../data/processed/ordenes_compra_licitacion_limpio.csv', index=False)